# 09: Boosting candidate discovery

Este notebook identifica candidatos de boosting usando validacion artificial. No usa test oficial.

Rol: comparar RandomForestRegressor como baseline contra XGBRegressor y LGBMRegressor si estan disponibles. Si no estan instalados, usa fallbacks sklearn claramente documentados.


In [ ]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results" / "FD001"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
VALIDATION_RANDOM_STATE = 42
CUT_RULS = (20, 50, 80, 110, 140)
BASE_WINDOW_SIZE = 30
BASE_RUL_CAP = 125

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [ ]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)
from src.fd001_experiment_utils import (
    add_bin_metric_columns,
    available_boosting_factories,
    fit_predict_model,
    lgbm_factory,
    metric_row_from_predictions,
    selection_sort,
)


In [ ]:
FORCE_RERUN = False
METRICS_PATH = RESULTS_DIR / "fd001_boosting_metrics.csv"
BINS_PATH = RESULTS_DIR / "fd001_boosting_metrics_by_rul_bin.csv"
PREDICTIONS_PATH = RESULTS_DIR / "fd001_boosting_predictions.csv"
FIGURE_DIR = FIGURES_DIR / "fd001_boosting"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


## Run or load


In [ ]:
if METRICS_PATH.exists() and BINS_PATH.exists() and PREDICTIONS_PATH.exists() and not FORCE_RERUN:
    metrics = pd.read_csv(METRICS_PATH)
    bin_metrics = pd.read_csv(BINS_PATH)
    predictions = pd.read_csv(PREDICTIONS_PATH)
    print("Se cargan resultados existentes de boosting.")
else:
    set_global_seed(VALIDATION_RANDOM_STATE)
    prepared = prepare_fd001_temporal_validation_only(
        data_dir=DATA_DIR,
        eval_size=EVAL_SIZE,
        random_state=VALIDATION_RANDOM_STATE,
        max_rul=BASE_RUL_CAP,
        cut_ruls=CUT_RULS,
        window_size=BASE_WINDOW_SIZE,
    )
    factories, notes = available_boosting_factories(random_state=VALIDATION_RANDOM_STATE)
    print("\n".join(notes))
    prediction_tables = []
    representation = f"temporal_w{BASE_WINDOW_SIZE}"
    for model_name, factory in factories.items():
        print(f"Entrenando {model_name}")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pred_df = fit_predict_model(prepared, factory(), model_name, representation)
        prediction_tables.append(pred_df)

    predictions = pd.concat(prediction_tables, ignore_index=True)
    metrics = metrics_by_model(predictions)
    metrics.insert(2, "window_size", BASE_WINDOW_SIZE)
    metrics.insert(3, "rul_cap", BASE_RUL_CAP)
    metrics.insert(4, "n_features", len(prepared["feature_columns"]))
    metrics.insert(5, "target_used_for_training", "RUL_capped")
    bin_metrics = metrics_by_rul_bin(predictions)
    metrics = add_bin_metric_columns(metrics, bin_metrics)

    metrics.to_csv(METRICS_PATH, index=False)
    bin_metrics.to_csv(BINS_PATH, index=False)
    predictions.to_csv(PREDICTIONS_PATH, index=False)

display(selection_sort(metrics))


## Lectura

LGBMRegressor y XGBRegressor son candidatos reales si estan disponibles. RandomForestRegressor queda como baseline temporal fuerte.
